# 05 Evaluation and honest reporting

**Question:** What evidence would establish that this recommender works? Separate software correctness, retrieval sanity, and user relevance. A perfect sanity check is not a perfect recommendation system.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / "book_recommender").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
import pandas as pd
from book_recommender.config import RAW, CLEAN, CLASSIFIED, ENRICHED, REPORTS


## Run the tests
The integration tests use real persisted Chroma with tiny deterministic embeddings to isolate ID mapping, filters, rebuilds, and stale-data detection. They do not validate pretrained model quality.

In [2]:
import subprocess
result = subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout)
assert result.returncode == 0, result.stderr

........                                                                 [100%]
8 passed in 1.54s



## Self retrieval and latency
Run `python -m scripts.evaluate`. It searches with sampled book descriptions and asks whether the source book ranks first. Timings exclude initial model loading. Near-duplicate descriptions can produce ties.

In [3]:
import json
path = REPORTS / "evaluation.json"
if path.exists():
    display(json.loads(path.read_text()))
else:
    print("Run python -m scripts.evaluate first.")

{'catalog_size': 5729,
 'sanity_queries': 20,
 'description_self_retrieval_hit_at_1': 1.0,
 'median_search_seconds': 0.09406050000688992,
 'p95_search_seconds': 0.33218233998632063,
 'interpretation': 'Self-retrieval checks indexing, not recommendation quality. Timing excludes model initialization.'}

## Human relevance labels
Inspect `data/evaluation_queries.json`. Fill relevant ISBNs by judging a pooled set of semantic and TF-IDF results without knowing which system retrieved them. Keep a separate held-out query set for final reporting. Empty labels are intentionally rejected by the evaluator.

In [4]:
judgments = json.loads((ROOT / "data/evaluation_queries.json").read_text())
pd.DataFrame(judgments)

,query,relevant_isbns
0,A story about unlikely friendships and finding...,[]
1,A detective solving a mysterious disappearance,[]
2,A family coping with grief and learning to for...,[]
3,Understanding stars and the origins of the uni...,[]
4,Practical advice for building better daily habits,[]
5,A journey through an unfamiliar world full of ...,[]
6,Understanding how the human brain and memory work,[]
7,A person seeking revenge and confronting its c...,[]
8,Learning to cook simple meals at home,[]
9,Historical accounts of political power and rev...,[]


## Metrics
Precision@5 is relevant retrieved books divided by five. Recall@5 is relevant retrieved books divided by all relevant books in the judged set. nDCG discounts lower ranks and divides by the ideal ranking. Incomplete relevance labels distort recall; do not claim dataset-wide accuracy from a small subjective test.

## Final demonstration
Start `python app.py`. Show a semantic query, category constraint, tone reranking, and score explanation. Report actual catalog size and measured latency from `reports/`; mention truncation, weak labels, and cold-start limitations. This is inference using pretrained models, not custom LLM training.